# Linear Regression Model Feature Engineering Phase

The purpose of this notebook is to engineer features to better predict the nighly pricing of airbnb listings in the Nashville area. There will be no imputation of continous variables, dummy creations, or scaling done as this will be saved until after the data is split into training and testing sets. 

### Load dataset and check first few rows

In [9]:
import json
import re

import pandas as pd
import numpy as np

# Tells pandas to display all columns when printing a dataframe
pd.set_option("display.max_columns", None)
df = pd.read_csv("../data/cleaned/listings_cleaned.csv")

print(df.shape)
df.head()

(8764, 15)


,host_is_superhost,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms_text,bedrooms,amenities,price_quote_price_per_night,minimum_nights,review_scores_rating,calculated_host_listings_count,has_license
0,t,District 25,36.124660,-86.81269,Private room in home,Private room,2,1 private bath,1.0,"[""Free street parking"", ""Lock on bedroom door""...",80.0,1.0,4.94,1,0
1,t,District 18,36.131220,-86.80066,Entire rental unit,Entire home/apt,2,1 bath,2.0,"[""Free street parking"", ""40 inch HDTV with Ama...",132.0,2.0,4.92,1,0
2,t,District 19,36.175800,-86.79950,Entire guesthouse,Entire home/apt,2,1 bath,1.0,"[""Free street parking"", ""Body soap"", ""Dishes a...",123.5,1.0,4.90,7,0
3,t,District 6,36.164668,-86.75054,Private room in home,Private room,2,1 private bath,1.0,"[""Free street parking"", ""Oven"", ""32 inch TV wi...",114.0,1.0,4.86,1,0
4,t,District 19,36.172300,-86.79250,Entire guest suite,Entire home/apt,2,1 bath,1.0,"[""Free street parking"", ""Body soap"", ""Dishes a...",128.0,1.0,4.95,7,0


### Location

Latitude and longitude are not useful to a linear model on their own, since price does not rise or fall in a straight line as you move north or east. What matters is how close a listing is to the main tourist area, so both are converted to a distance from Lower Broadway.

Haversine distance is used because the Earth is curved, and one degree of longitude is shorter than one degree of latitude at Nashville's latitude.

In [10]:
# Lower Broadway coordinates, the center of Nashville's tourist area
BROADWAY_LAT = 36.1612
BROADWAY_LON = -86.7775
EARTH_RADIUS_KM = 6371


def haversine_km(lat1, lon1, lat2, lon2):
    # Trig functions in numpy expect radians, not degrees
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))


df["distance_to_broadway_km"] = haversine_km(df["latitude"], df["longitude"], BROADWAY_LAT, BROADWAY_LON)
df["distance_to_broadway_km"].describe()

count    8764.000000
mean        4.469349
std         3.710419
min         0.045331
25%         2.059317
50%         3.601944
75%         5.635577
max        27.588098
Name: distance_to_broadway_km, dtype: float64

#### Distance Bands

The distance is split into bands so the model can give each area its own price effect instead of assuming that every extra kilometer changes price by the same amount. Each cutoff follows a real change in the city:

- **0-3 km:** the downtown core, including SoBro, the Gulch, and Germantown. Lower Broadway is a short walk or ride away.
- **3-5 km:** the inner neighborhoods just outside downtown, such as 12 South and Belmont.
- **5-10 km:** the wider urban area, such as Sylvan Park, Donelson, and Opryland. Reaching downtown usually means driving.
- **10+ km:** the outer suburbs, such as Madison, Antioch, and Hermitage.

In [11]:
# right=False makes each band include its lower cutoff, so a listing at exactly 3 km falls in 3-5 km
df["distance_band"] = pd.cut(
    df["distance_to_broadway_km"],
    bins=[0, 3, 5, 10, np.inf],
    labels=["0-3 km", "3-5 km", "5-10 km", "10+ km"],
    right=False,
)

# Check that every listing landed in a band
print("Listings without a band:", df["distance_band"].isnull().sum())
df["distance_band"].value_counts().sort_index()

Listings without a band: 0


distance_band
0-3 km     3636
3-5 km     2556
5-10 km    1955
10+ km      617
Name: count, dtype: int64

### Amenities

The amenities column stores each listing's amenities as a JSON list in a text string, so it has to be parsed before anything can be counted. The labels are free-form, with variants like "Shared outdoor pool - heated", so each flag uses a regular expression rather than an exact match.

In [12]:
# Parse the text into a Python list for each listing
amenities_list = df["amenities"].apply(json.loads)

df["amenity_count"] = amenities_list.apply(len)
df["amenity_count"].describe()

count    8764.000000
mean       45.726495
std        13.809579
min         0.000000
25%        37.000000
50%        47.000000
75%        55.000000
max       100.000000
Name: amenity_count, dtype: float64

In [13]:
# A plain search for "pool" would also match "Pool table", "Pool view", and appliances like
# "Whirlpool refrigerator", so the pool pattern only accepts labels that are actually a pool.
# The gym pattern skips anything nearby, since a gym down the street is not part of the listing.
# Exercise equipment does not count as a gym.
# Free parking includes garage and driveway labels, which are also free and on site.
# Only 8 listings use the "Game room" label, so game room also counts the equipment Airbnb lists
# separately. Board games and consoles are left out because they do not suggest a dedicated room.
amenity_patterns = {
    "has_pool": r"^((private|shared) )?((indoor|outdoor) )?pool( -|$)",
    "has_hot_tub": r"hot tub",
    "has_gym": r"\bgym\b(?! nearby)",
    "has_game_room": r"^(game room|ping pong table|pool table|arcade games|life size games)$",
    "has_fire_pit": r"^fire pit$",
    "has_ev_charger": r"^ev charger",
    "pets_allowed": r"^pets allowed$",
    "has_city_skyline_view": r"^city skyline view$",
    "has_free_parking": r"^free .*parking.* on premises",
}

for column, pattern in amenity_patterns.items():
    df[column] = amenities_list.apply(
        lambda items: int(any(re.search(pattern, item, re.IGNORECASE) for item in items))
    )

# Check how many listings have each amenity
df[list(amenity_patterns)].sum().sort_values(ascending=False)

has_free_parking         6691
pets_allowed             2345
has_fire_pit             2153
has_pool                 2117
has_gym                  1797
has_game_room            1336
has_city_skyline_view    1263
has_hot_tub              1061
has_ev_charger            592
dtype: int64

### Category Grouping

Only meaning-based grouping is done here. These rules look at one row at a time and would be the same no matter which listings are in the data, so they cannot leak information from the test set. Frequency-based grouping, like merging rare values of `neighbourhood_cleansed`, depends on how common each category is and has to wait until after the split.

#### Room Type

In [14]:
# Entire home/apt makes up about 92% of listings, and Shared room only has 5
# The real pricing split is whether the guest gets the whole place or just a room
df["room_type_grouped"] = np.where(df["room_type"] == "Entire home/apt", "Entire place", "Room")

pd.crosstab(df["room_type"], df["room_type_grouped"])

room_type_grouped,Entire place,Room
room_type,,
Entire home/apt,8091,0
Hotel room,0,73
Private room,0,595
Shared room,0,5


#### Property Type

In [15]:
# Room details are already captured by room_type_grouped, so "Private room in home" and
# "Entire home" both reduce to the building type "home"
def building_type(property_type):
    value = property_type.lower()
    for prefix in ["private room in ", "shared room in ", "room in ", "entire "]:
        value = value.removeprefix(prefix)
    return value


property_groups = {
    "home": "House",
    "bungalow": "House",
    "cottage": "House",
    "villa": "House",
    "vacation home": "House",
    "casa particular": "House",
    "rental unit": "Apartment/Condo",
    "condo": "Apartment/Condo",
    "loft": "Apartment/Condo",
    "serviced apartment": "Apartment/Condo",
    "townhouse": "Townhouse",
    "hotel": "Hotel/Resort",
    "boutique hotel": "Hotel/Resort",
    "aparthotel": "Hotel/Resort",
    "resort": "Hotel/Resort",
    "bed and breakfast": "Hotel/Resort",
    "guest suite": "Guest Unit",
    "guesthouse": "Guest Unit",
    "tiny home": "Other",
    "cabin": "Other",
    "camper/rv": "Other",
    "barn": "Other",
    "farm stay": "Other",
    "nature lodge": "Other",
    "treehouse": "Other",
    "boat": "Other",
    "dome": "Other",
    # "Entire place" and "Private room" do not name a building type
    "place": "Other",
    "private room": "Other",
}

df["property_group"] = df["property_type"].apply(building_type).map(property_groups)

# Check that every property type was mapped to a group
print("Listings without a group:", df["property_group"].isnull().sum())
df["property_group"].value_counts()

Listings without a group: 0


property_group
House              3476
Apartment/Condo    3409
Townhouse           864
Hotel/Resort        504
Guest Unit          431
Other                80
Name: count, dtype: int64

### Target

In [16]:
# Nightly prices are heavily right-skewed, so log_price is kept next to the raw price as a less skewed target
# Listings with no price stay missing here. Both target columns must be dropped from X in modeling.
df["log_price"] = np.log(df["price_quote_price_per_night"])

df[["price_quote_price_per_night", "log_price"]].describe()

,price_quote_price_per_night,log_price
count,8201.000000,8201.000000
mean,416.372870,5.798634
std,469.098751,0.624111
min,26.600000,3.280911
25%,221.000000,5.398163
50%,320.000000,5.768321
75%,462.380000,6.136387
max,18708.000000,9.836707


### Remove Replaced Columns

In [17]:
# Each of these has been replaced by an engineered feature
df = df.drop(columns=[
    "latitude",
    "longitude",
    "distance_to_broadway_km",
    "amenities",
    "room_type",
    "property_type",
])

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8764 entries, 0 to 8763
Data columns (total 24 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   host_is_superhost               8764 non-null   str     
 1   neighbourhood_cleansed          8764 non-null   str     
 2   accommodates                    8764 non-null   int64   
 3   bathrooms_text                  8764 non-null   str     
 4   bedrooms                        8761 non-null   float64 
 5   price_quote_price_per_night     8201 non-null   float64 
 6   minimum_nights                  8714 non-null   float64 
 7   review_scores_rating            7849 non-null   float64 
 8   calculated_host_listings_count  8764 non-null   int64   
 9   has_license                     8764 non-null   int64   
 10  distance_band                   8764 non-null   category
 11  amenity_count                   8764 non-null   int64   
 12  has_pool                       

### View Dataset After Feature Engineering

In [18]:
print(df.shape)
df.head()

(8764, 24)


,host_is_superhost,neighbourhood_cleansed,accommodates,bathrooms_text,bedrooms,price_quote_price_per_night,minimum_nights,review_scores_rating,calculated_host_listings_count,has_license,distance_band,amenity_count,has_pool,has_hot_tub,has_gym,has_game_room,has_fire_pit,has_ev_charger,pets_allowed,has_city_skyline_view,has_free_parking,room_type_grouped,property_group,log_price
0,t,District 25,2,1 private bath,1.0,80.0,1.0,4.94,1,0,5-10 km,27,0,0,0,0,0,0,0,0,1,Room,House,4.382027
1,t,District 18,2,1 bath,2.0,132.0,2.0,4.92,1,0,3-5 km,42,0,0,0,0,0,0,0,0,0,Entire place,Apartment/Condo,4.882802
2,t,District 19,2,1 bath,1.0,123.5,1.0,4.90,7,0,0-3 km,43,0,0,0,0,0,0,0,0,1,Entire place,Guest Unit,4.816241
3,t,District 6,2,1 private bath,1.0,114.0,1.0,4.86,1,0,0-3 km,29,0,0,0,0,0,0,0,0,1,Room,House,4.736198
4,t,District 19,2,1 bath,1.0,128.0,1.0,4.95,7,0,0-3 km,42,0,0,0,0,0,0,0,1,1,Entire place,Guest Unit,4.852030


### Pricing Outliers

The highest nightly prices are mostly large houses that sleep 16 guests, so a price alone does not show whether a listing is real. A $7,000 night is believable for a 16-guest estate but not for a 4-guest private room. Price per guest is used instead, since it adjusts for the size of the listing. The typical listing costs about $48 per guest.

- **Flag:** Listings at $400 or more per guest, so they can be reviewed.
- **Removal:** listings at $1,000 or more per guest are dropped as likely data errors. This removes a $18,708 house and a $5,490 private room that sleeps 4.
- **Kept but flagged:** the other flagged listings are large luxury houses and stay in the data.

The flag is built from price, which is the target, so it must be dropped from X in modeling, like the two price columns.

In [ ]:
# Price per guest adjusts for listing size: a 16-guest estate can cost far more than a 2-guest room
price_per_guest = df["price_quote_price_per_night"] / df["accommodates"]

# Flag listings at $400+ per guest for review.
df["price_outlier_flag"] = (price_per_guest >= 400).astype(int)

print("Flagged listings:", df["price_outlier_flag"].sum())
flagged = df.loc[df["price_outlier_flag"] == 1, [
    "price_quote_price_per_night", "accommodates", "bedrooms", "room_type_grouped", "property_group"
]].assign(price_per_guest=price_per_guest.round(0))
print(flagged.sort_values("price_per_guest", ascending=False).to_string())

# Remove the obvious errors: $1,000+ per guest. Listings with no price are kept
too_high = price_per_guest >= 1000
print("Removed listings:", too_high.sum())
df = df[~too_high]

print(df.shape)

Flagged listings: 13
      price_quote_price_per_night  accommodates  bedrooms room_type_grouped   property_group  price_per_guest
2318                      1750.00             1       1.0      Entire place  Apartment/Condo           1750.0
1563                      5490.00             4       1.0              Room            Other           1372.0
7795                     18708.00            16      16.0      Entire place            House           1169.0
24                        1148.00             1       2.0              Room            House           1148.0
7748                       872.18             1      12.0      Entire place            House            872.0
7000                      8693.67            16      20.0      Entire place            House            543.0
8045                      8000.00            16      12.0      Entire place            House            500.0
3326                      2869.00             6       2.0      Entire place  Apartment/Condo       

### Save Feature Dataset

In [21]:
# Save the feature dataset for the modeling notebook
# index=False keeps pandas from writing the row index as an extra column
df.to_csv("../data/cleaned/listings_features.csv", index=False)